# 08 - MCP Hibrido con Grafos (LangGraph)

Objetivo: construir un flujo con grafo para enrutar preguntas del Mundial hacia:
- Nodo SQL (PostgreSQL)
- Nodo Vector (ChromaDB)

Arquitectura:

Pregunta -> Router (LLM) -> [sql_node | vector_node] -> END

In [12]:
# Basic imports
import json
import re
from pathlib import Path
from typing import Any, Literal, TypedDict

import chromadb
import psycopg2
from langchain_ollama import ChatOllama, OllamaEmbeddings
from langgraph.graph import END, StateGraph
from fastmcp import FastMCP

In [13]:
# Conexiones
conn = psycopg2.connect(
    host="localhost",
    database="WorldCup",
    user="guane",
    password="tu_password",
    port="5432"
)

PROJECT_ROOT = Path.cwd().parents[1]
VECTOR_DB_PATH = PROJECT_ROOT / "data" / "vector_db_2"
COLLECTION_NAME = "mundiales_football"

client = chromadb.PersistentClient(path=str(VECTOR_DB_PATH))
collection = client.get_collection(COLLECTION_NAME)
embeddings = OllamaEmbeddings(model="nomic-embed-text:latest")

print("DB SQL conectada")
print("Colecciones vectoriales:", [c.name for c in client.list_collections()])

DB SQL conectada
Colecciones vectoriales: ['mundiales_football']


In [14]:
# Tools base
def sql_matches_by_year(year: int, limit: int = 5):
    query = """
    SELECT year, datetime, hometeamname, hometeamgoals, awayteamgoals, awayteamname
    FROM world_cup_matches_raw
    WHERE year = %s
    ORDER BY datetime
    LIMIT %s
    """
    with conn.cursor() as cursor:
        cursor.execute(query, (str(year), limit))
        rows = cursor.fetchall()

    columns = ["year", "datetime", "home_team", "home_goals", "away_goals", "away_team"]
    return [dict(zip(columns, row)) for row in rows]


def vector_search_worldcup(question: str, n_results: int = 3):
    query_embedding = embeddings.embed_query(question)
    result = collection.query(query_embeddings=[query_embedding], n_results=n_results)
    docs = result.get("documents", [[]])[0]
    metas = result.get("metadatas", [[]])[0]

    return [
        {
            "rank": i + 1,
            "text": doc,
            "metadata": meta or {}
        }
        for i, (doc, meta) in enumerate(zip(docs, metas))
    ]

In [15]:
# Grafo + Prompts + text2sql
router_llm = ChatOllama(model="llama3.2:3b", temperature=0)

ROUTER_PROMPT = """
Eres un router para un asistente de la Copa del Mundo.
Tu tarea es decidir la fuente correcta de datos.

Responde SOLO con una palabra: sql o vector.

Reglas:
- sql: partidos, resultados, marcadores, goles, conteos por año.
- vector: historia, contexto, analisis, explicaciones narrativas.
- Si hay un año explicito (1930, 2014, etc.) y piden partidos/resultados, usa sql.
""".strip()

TEXT2SQL_PROMPT = """
Convierte la pregunta del usuario en UNA consulta SQL valida para PostgreSQL.

Restricciones estrictas:
- Responde solo SQL, sin markdown ni explicaciones.
- Solo SELECT.
- Usa solo esta tabla: world_cup_matches_raw.
- Si aplica, usa LIMIT 10 por defecto.

Columnas utiles:
year, datetime, stage, stadium, city,
hometeamname, hometeamgoals, awayteamgoals, awayteamname,
attendance, referee
""".strip()

ANSWER_PROMPT = """
Eres un asistente del Mundial.
Responde SOLO usando los datos entregados en CONTEXTO.
Si el contexto no alcanza, dilo explicitamente.
No inventes informacion.
""".strip()


class GraphState(TypedDict):
    question: str
    route: Literal["sql", "vector", "unknown"]
    result: Any
    answer: str


def detect_year(text: str):
    match = re.search(r"(19|20)\d{2}", text or "")
    return int(match.group(0)) if match else None


def route_with_llm(question: str) -> str:
    decision = (
        router_llm.invoke(f"{ROUTER_PROMPT}\n\nPregunta: {question}\nRuta:")
        .content
        .strip()
        .lower()
    )
    return "sql" if "sql" in decision else "vector"


def text2sql(question: str) -> str:
    raw_sql = (
        router_llm.invoke(f"{TEXT2SQL_PROMPT}\n\nPregunta: {question}\nSQL:")
        .content
        .strip()
    )

    sql = raw_sql.replace("```sql", "").replace("```", "").strip()
    low = sql.lower()

    # Guardia minima para mantenerla de solo lectura y en la tabla permitida.
    blocked = ["insert", "update", "delete", "drop", "alter", "truncate", "create"]
    if any(token in low for token in blocked):
        low = ""

    if not low.startswith("select") or "world_cup_matches_raw" not in low:
        year = detect_year(question)
        if year is not None:
            return (
                "SELECT year, datetime, hometeamname, hometeamgoals, awayteamgoals, awayteamname "
                "FROM world_cup_matches_raw "
                f"WHERE year = '{year}' ORDER BY datetime LIMIT 10"
            )
        return (
            "SELECT year, datetime, hometeamname, hometeamgoals, awayteamgoals, awayteamname "
            "FROM world_cup_matches_raw ORDER BY year, datetime LIMIT 10"
        )

    if "limit" not in low:
        sql = f"{sql.rstrip(';')} LIMIT 10"

    return sql


# MCP tools (usadas por los nodos del grafo)
mcp = FastMCP("WorldCup Graph MCP")


@mcp.tool()
def mcp_sql_tool(question: str):
    sql_query = text2sql(question)

    try:
        with conn.cursor() as cursor:
            cursor.execute(sql_query)
            rows = cursor.fetchall()
            columns = [desc[0] for desc in cursor.description]
        data = [dict(zip(columns, row)) for row in rows]
        return {"sql_query": sql_query, "rows": data}
    except Exception as exc:
        # Fallback robusto para evitar UndefinedFunction u otros errores de SQL generado.
        conn.rollback()
        year = detect_year(question)
        safe_sql = (
            "SELECT year, datetime, hometeamname, hometeamgoals, awayteamgoals, awayteamname "
            "FROM world_cup_matches_raw "
            + (f"WHERE year = '{year}' " if year is not None else "")
            + "ORDER BY year, datetime LIMIT 10"
        )
        try:
            with conn.cursor() as cursor:
                cursor.execute(safe_sql)
                rows = cursor.fetchall()
                columns = [desc[0] for desc in cursor.description]
            data = [dict(zip(columns, row)) for row in rows]
            return {"sql_query": safe_sql, "rows": data, "warning": f"Fallback SQL por error: {exc}"}
        except Exception as exc2:
            conn.rollback()
            return {
                "sql_query": safe_sql,
                "rows": [],
                "warning": f"No se pudo ejecutar SQL generado ({exc}) ni fallback ({exc2})"
            }


@mcp.tool()
def mcp_vector_tool(question: str, n_results: int = 3):
    return vector_search_worldcup(question=question, n_results=n_results)


def router_node(state: GraphState) -> GraphState:
    route = route_with_llm(state["question"])
    return {**state, "route": route}


def sql_node(state: GraphState) -> GraphState:
    data = mcp_sql_tool(question=state["question"])
    return {**state, "result": data, "route": "sql"}


def vector_node(state: GraphState) -> GraphState:
    data = mcp_vector_tool(question=state["question"], n_results=3)
    return {**state, "result": data, "route": "vector"}


def answer_node(state: GraphState) -> GraphState:
    context = json.dumps(state["result"], ensure_ascii=False)
    answer = router_llm.invoke(
        f"{ANSWER_PROMPT}\n\nPregunta: {state['question']}\n\nCONTEXTO:\n{context}\n\nRespuesta:"
    ).content
    return {**state, "answer": str(answer).strip()}


def branch_after_router(state: GraphState) -> str:
    return state.get("route", "vector")


builder = StateGraph(GraphState)
builder.add_node("router", router_node)
builder.add_node("sql", sql_node)
builder.add_node("vector", vector_node)
builder.add_node("answer", answer_node)
builder.set_entry_point("router")
builder.add_conditional_edges("router", branch_after_router, {"sql": "sql", "vector": "vector"})
builder.add_edge("sql", "answer")
builder.add_edge("vector", "answer")
builder.add_edge("answer", END)

graph = builder.compile()
print("Grafo compilado ✅")
print("Incluye: router prompt + text2sql + respuesta con contexto")

Grafo compilado ✅
Incluye: router prompt + text2sql + respuesta con contexto


## Visualizar grafo

Si el entorno soporta Mermaid, esta celda imprime el diagrama del grafo.

In [16]:
from IPython.display import Image, display

# Visualiza el grafo como imagen PNG (estilo LangGraph)
display(Image(graph.get_graph().draw_mermaid_png()))

---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	router(router)
	sql(sql)
	vector(vector)
	answer(answer)
	__end__([<p>__end__</p>]):::last
	__start__ --> router;
	router -.-> sql;
	router -.-> vector;
	sql --> answer;
	vector --> answer;
	answer --> __end__;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc



```mermaid
---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	router(router)
	sql(sql)
	vector(vector)
	answer(answer)
	__end__([<p>__end__</p>]):::last
	__start__ --> router;
	router -.-> sql;
	router -.-> vector;
	sql --> answer;
	vector --> answer;
	answer --> __end__;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc

```

In [17]:
# Pruebas
questions = [
    "Muestrame partidos del mundial de 1930",
    "Explica por que fue importante el primer mundial",
    "Resultados del mundial 2014",
    "Como evoluciono tacticamente la copa del mundo"
]

for q in questions:
    print("\n" + "=" * 80)
    print("Pregunta:", q)
    try:
        # Limpia estado de transaccion si hubo error SQL previo.
        conn.rollback()

        out = graph.invoke({"question": q, "route": "unknown", "result": None, "answer": ""})
        print("Ruta final:", out["route"])

        if out["route"] == "sql":
            print("Tool MCP usada: mcp_sql_tool")
            print("SQL generado:", out["result"].get("sql_query", ""))
            rows = out["result"].get("rows", [])
            print("Preview SQL:", json.dumps(rows[:1], indent=2, ensure_ascii=False))
        else:
            print("Tool MCP usada: mcp_vector_tool")
            print("Preview Vector:", json.dumps(out["result"][:1], indent=2, ensure_ascii=False))

        print("Respuesta final:")
        print(out["answer"])
    except Exception as exc:
        conn.rollback()
        print("Error en la ejecucion del grafo:", exc)
        print("Sugerencia: reejecuta desde la celda de conexiones (cell 2) hasta esta celda.")


Pregunta: Muestrame partidos del mundial de 1930


Ruta final: sql
Tool MCP usada: mcp_sql_tool
SQL generado: SELECT year, datetime, hometeamname, hometeamgoals, awayteamgoals, awayteamname FROM world_cup_matches_raw WHERE year = '1930' ORDER BY year, datetime LIMIT 10
Preview SQL: [
  {
    "year": "1930",
    "datetime": "13 Jul 1930 - 15:00 ",
    "hometeamname": "France",
    "hometeamgoals": "4",
    "awayteamgoals": "1",
    "awayteamname": "Mexico"
  }
]
Respuesta final:
No tengo suficiente información para proporcionarte los partidos del Mundial de 1930. El contexto solo proporciona 10 partidos, pero no hay una forma clara de determinar qué otros partidos se jugaron en ese año.

Si deseas obtener la lista completa de partidos del Mundial de 1930, necesitaría más información o acceso a los datos completos del Mundial.

Pregunta: Explica por que fue importante el primer mundial
Ruta final: vector
Tool MCP usada: mcp_vector_tool
Preview Vector: [
  {
    "rank": 1,
    "text": "Historia  de  los  Mundiales  de  Fútbol  \nUn  análi

In [18]:
# Cerrar conexion SQL al finalizar (opcional)
# conn.close()